In [1]:
# Set up Python path
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path configured ✓")

Project root: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data
Python path configured ✓


In [2]:
# Import pipeline modules
from src.config import setup_experiment
from pipeline.preprocessing_stage import preprocessing_stage
from pipeline.trtr_stage import trtr_stage
from pipeline.cvae_stage import cvae_stage
from pipeline.tstr_stage import tstr_stage
from pipeline.evaluation_stage import evaluation_stage

print("✓ All pipeline modules imported successfully")

✓ All pipeline modules imported successfully


In [3]:
# Setup experiment
config_name = 'exp_102'
config, output_dir = setup_experiment(config_name)

print(f"\n{'='*80}")
print(f"EXPERIMENT CONFIGURATION")
print(f"{'='*80}")
print(f"\nExperiment: {config['name']}")
print(f"Description: {config['description']}")
print(f"Output directory: {output_dir}")
print(f"\nKey paths:")
print(f"  • Raw data: {config['paths']['raw_data_dir']}")
print(f"  • Output HDF5: {config['paths']['processed_file']}")
print(f"  • Models: {config['paths']['models_baseline_dir']}")
print(f"  • Synthetic: {config['paths']['synthetic_dir']}")
print(f"  • Evaluation: {config['paths']['statistical_dir']}")


EXPERIMENT CONFIGURATION

Experiment: exp_102
Description: Enhanced preprocessing: 3x3 spatial median filter + pre-computed patient tumor_masks
Output directory: results\exp_102

Key paths:
  • Raw data: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\raw
  • Output HDF5: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_102\data\glioma_dataset.h5
  • Models: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_102\models\baselines
  • Synthetic: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_102\synthetic
  • Evaluation: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_102\statistical_eval


In [8]:
# print working directory
%cd ..
print(f"\nCurrent working directory: {Path.cwd()}")

c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

Current working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data


In [9]:
"""
DRY-RUN: Verify tumor_maps.npy before preprocessing
Shows exactly which raw files have corresponding tumor_masks, 
and which don't. Catches any mismatches early.
"""
import numpy as np
from pathlib import Path
import glob

print(f"\n{'='*80}")
print("DRY-RUN: TUMOR MAPS VALIDATION")
print(f"{'='*80}\n")

# Step 1: Load tumor_maps.npy (if it exists)
tumor_maps_path = Path('results/exp_102/tumor_maps.npy')

if not tumor_maps_path.exists():
    print(f"⚠ tumor_maps.npy not found at {tumor_maps_path}")
    print(f"   Run cell above first\n")
else:
    tumor_maps = np.load(tumor_maps_path, allow_pickle=True).item()
    print(f"✓ Loaded tumor_maps.npy")
    print(f"  Contains {len(tumor_maps)} entries")
    print(f"  Sample keys: {list(tumor_maps.keys())[:3]}\n")
    
    # Step 2: Get all raw files
    raw_files = sorted(glob.glob('data/raw/LGm-*/*.npy'))
    print(f"✓ Found {len(raw_files)} raw files\n")
    
    # Step 3: Check each raw file
    has_map = 0
    no_map = 0
    shape_mismatch = 0
    
    print(f"{'FILE':50} {'RAW SHAPE':15} {'MAP SHAPE':15} {'STATUS':15}")
    print(f"{'-'*50} {'-'*15} {'-'*15} {'-'*15}")
    
    for raw_file in raw_files:
        filename = Path(raw_file).name  # e.g., "HF-988_V1-T_1.h5_13.npy"
        
        # Load raw file shape
        raw_data = np.load(raw_file)
        raw_shape = raw_data.shape[:2]
        
        # Check if in tumor_maps
        if filename in tumor_maps:
            tumor_mask = tumor_maps[filename]
            map_shape = tumor_mask.shape
            
            if raw_shape == map_shape:
                status = "✓ MATCH"
                has_map += 1
            else:
                status = f"✗ SHAPE MISMATCH"
                shape_mismatch += 1
            
            print(f"{filename:50} {str(raw_shape):15} {str(map_shape):15} {status:15}")
        else:
            status = "⚠ NO MAP"
            no_map += 1
            print(f"{filename:50} {str(raw_shape):15} {'':15} {status:15}")
    
    print(f"\n{'='*80}")
    print("SUMMARY:")
    print(f"  ✓ Matched (correct): {has_map}")
    print(f"  ⚠ No map found:      {no_map}")
    print(f"  ✗ Shape mismatch:    {shape_mismatch}")
    print(f"  Total raw files:     {len(raw_files)}")
    print(f"{'='*80}\n")
    
    if no_map > 0 or shape_mismatch > 0:
        print("⚠ WARNING: Issues found! Do NOT proceed to preprocessing.")
        print("   Verify tumor_maps.npy is complete and correct.\n")
    else:
        print("✓ All raw files have matching tumor_maps with correct shapes!")
        print("  Ready to proceed to preprocessing.\n")


DRY-RUN: TUMOR MAPS VALIDATION

✓ Loaded tumor_maps.npy
  Contains 59 entries
  Sample keys: ['HF-1002_V1AT_1.h5_0.npy', 'HF-1002_V2AT_1.h5_1.npy', 'HF-1010_V1T_1.h5_0.npy']

✓ Found 59 raw files

FILE                                               RAW SHAPE       MAP SHAPE       STATUS         
-------------------------------------------------- --------------- --------------- ---------------
HF-1293_13_1.h5_0.npy                              (247, 31)       (247, 31)       ✓ MATCH        
HF-1295_V3AM_2.h5_1.npy                            (245, 61)       (245, 61)       ✓ MATCH        
HF-2534_V2B_1.h5_2.npy                             (64, 64)        (64, 64)        ✓ MATCH        
HF-448_V5B_1.h5_3.npy                              (60, 60)        (60, 60)        ✓ MATCH        
HF-868_1_2.h5_4.npy                                (119, 59)       (119, 59)       ✓ MATCH        
HF-1010_V1T_1.h5_0.npy                             (60, 60)        (60, 60)        ✓ MATCH        
HF-1016_IA

In [10]:
print("\n" + "="*80)
print("STAGE 1: PREPROCESSING (1-2 hours)")
print("="*80)

output_h5 = preprocessing_stage(config, output_dir)
print(f"\n✓ Stage 1 complete: {output_h5}")


STAGE 1: PREPROCESSING (1-2 hours)

PREPROCESSING STAGE
Input:  C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\raw
Output: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_102\data\glioma_dataset.h5
Calibration Loaded. Range: 50.0 - 3399.8 cm⁻¹

Configuration:
  • Spatial median filter: True
  • Spatial smoothing: False
  • Quality filtering: False
  • Cosmic ray removal: False
  • Tumor mask source: results/exp_102/tumor_maps.npy

Starting preprocessing...
Found 59 scan files.
Loading Tumor Maps from results\exp_102\tumor_maps.npy...
Legacy Mode: Output shape is 1738 features (Full spectrum, no cropping).


Processing:   0%|          | 0/59 [00:00<?, ?it/s]

Applying spatial median filter (size=3×3)...


Processing:   2%|▏         | 1/59 [00:49<47:23, 49.02s/it]

Applying spatial median filter (size=3×3)...


Processing:   3%|▎         | 2/59 [01:52<54:53, 57.77s/it]

... (110 outputs left out)

Applying spatial median filter (size=3×3)...


Processing: 100%|██████████| 59/59 [43:48<00:00, 44.56s/it]



Processing complete. Saved to C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\exp_102\data\glioma_dataset.h5

QUALITY FILTER STATISTICS:
  Total spectra processed:  226,284
  Passed quality checks:    226,284 (100.0%)
  Rejected as junk:         0 (0.0%)


RuntimeError: Unable to synchronously get dataspace (identifier is not of specified type)

In [ ]:
print("\n" + "="*80)
print("STAGE 2: TRTR (Train on Real, Test on Real)")
print("="*80)

trtr_results = trtr_stage(config, output_dir)
print(f"\n✓ Stage 2 complete")

In [ ]:
print("\n" + "="*80)
print("STAGE 3: CVAE (Conditional VAE - Train Generative Model)")
print("="*80)

cvae_results = cvae_stage(config, output_dir)
print(f"\n✓ Stage 3 complete")

In [ ]:
print("\n" + "="*80)
print("STAGE 4: TSTR (Train on Synthetic, Test on Real)")
print("="*80)

tstr_results = tstr_stage(config, output_dir, cvae_results)
print(f"\n✓ Stage 4 complete")

In [ ]:
print("\n" + "="*80)
print("STAGE 5: EVALUATION (Bootstrap + Patient-Stratified Metrics)")
print("="*80)

eval_results = evaluation_stage(trtr_results, tstr_results, config, output_dir)
print(f"\n✓ Stage 5 complete")

In [ ]:
print("\n" + "="*80)
print("PIPELINE EXECUTION COMPLETE ✓")
print("="*80)

print(f"\nOutput directory: {output_dir}")
print(f"\nGenerated artifacts:")
print(f"  ✓ Preprocessed data: {output_dir}/data/glioma_dataset.h5")
print(f"  ✓ TRTR models: {output_dir}/models/baselines/")
print(f"  ✓ CVAE models: {output_dir}/models/cvae/")
print(f"  ✓ Synthetic data: {output_dir}/synthetic/synthetic_spectra.npz")
print(f"  ✓ TSTR models: {output_dir}/models/tstr/")
print(f"  ✓ Evaluation: {output_dir}/statistical_eval/")
print(f"\n{'='*80}")